In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
# Q1 solution
len(documents)

72

In [5]:
# Q2 solution
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index


idx = build_index(documents)

In [6]:
search_res = idx.search("How does the agentic loop keep calling the model until it stops?")

In [18]:
q2_filename=search_res[0]["filename"]
print(q2_filename)

01-agentic-rag/lessons/14-agentic-loop.md


In [11]:
#Q3 solution

from rag_helper import RAGBase

In [14]:
search_res[0].keys()

dict_keys(['content', 'filename'])

In [53]:
class RAGHW1(RAGBase):

    # using filename as an alternative of self.course

    def search(self, query, num_results=3):
        boost_dict = {'question': 3.0, 'section': 0.5}
        filter_dict = {'filename': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
    
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()
    
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage



In [20]:
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()


True

In [54]:
raghw = RAGHW1(index=idx, llm_client=openai_client, course=q2_filename)

In [55]:
q3 = raghw.rag("How does the agentic loop keep calling the model until it stops?")

In [58]:
q3[1]

ResponseUsage(input_tokens=2326, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2438)

In [59]:
#Q4

In [60]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [61]:
len(chunks)

295

In [65]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [66]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [68]:
idx_chunked = build_index(chunks)

In [69]:
#Q5 
raghw = RAGHW1(index=idx_chunked, llm_client=openai_client, course=q2_filename)
q5 = raghw.rag("How does the agentic loop keep calling the model until it stops?")

In [71]:
q5[1]

ResponseUsage(input_tokens=1436, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=121, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1557)

In [72]:
1436/(2326+1792)

0.348712967459932